# Assignment 8: Build the Intent Dataset for Your Own Chatbot
### AITCD001 Chatbots Development · Learning Outcome 3 · United African Institute of Technology

**Due Monday 02nd November 2026, 23:59**

Cells marked **YOUR DOMAIN** are replaced with your own chatbot's data. Cells marked **PROVIDED**
do the checking, the split, the baseline and the entity count: read them, do not change them.

The worked example is a **fictional** bus ticket desk (the Week 7 example) with only 6 messages
per label, so the quality report shows what is still missing. It may not be submitted.

**Steps:** Save a copy of this file in Drive. Replace Steps 1, 2, 3, 6 and 7 with your own, and run
everything from the top (Runtime, then Run all). Repeat Steps 3 and 4 until the report is clean.

---

---

## Setup

**PROVIDED.** Run once. The same spaCy model as the session notebook.

In [1]:
# AITCD001, Week 8 take home: the intent dataset for my own chatbot
import csv
import random
import re
from collections import Counter

import pandas as pd
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:                  # not installed yet: install it once
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("ready: spaCy", spacy.__version__, "| model", nlp.meta["version"])

ready: spaCy 3.8.18 | model 3.8.0


---

## Step 1: The chatbot and its labels

**YOUR DOMAIN.** The chatbot of your Portfolio Project 1. All of its intents, plus `out_of_scope`. Do not add `greeting` or `name` unless your users really ask for something with them.

In [2]:
# STEP 1. YOUR DOMAIN: the chatbot of your Portfolio Project 1, and its labels.
BOT_NAME = "Exam Attendance Tracker"
INTENTS = ["sign_in", "sign_out", "check_status", "venue_location", "help", "out_of_scope"]
TARGET = 30          # messages per label, at least
print(BOT_NAME, "|", len(INTENTS), "labels:", INTENTS)

Exam Attendance Tracker | 6 labels: ['sign_in', 'sign_out', 'check_status', 'venue_location', 'help', 'out_of_scope']


---

## Step 2: The labelling guideline

**YOUR DOMAIN.** Write the guideline BEFORE labelling. Every time two labels are possible for a message, add or sharpen a rule.

In [3]:
# STEP 2. YOUR DOMAIN: your labelling guideline.
GUIDELINE = [
    "One message, one label. Two requests in one message: split it first.",
    "Label the goal, not the words: 'where do I sign in for my exam' is venue_location.",
    "Any action regarding arriving or entering an exam hall is sign_in.",
    "Any action regarding leaving or finishing an exam session is sign_out.",
    "System errors, general help, complaints, or external queries: out_of_scope.",
    "Keep the student's exact words: typos, SMS, Kinyarwanda, as typed.",
]
for i, rule in enumerate(GUIDELINE, start=1):
    print(i, rule)

1 One message, one label. Two requests in one message: split it first.
2 Label the goal, not the words: 'where do I sign in for my exam' is venue_location.
3 Any action regarding arriving or entering an exam hall is sign_in.
4 Any action regarding leaving or finishing an exam session is sign_out.
5 System errors, general help, complaints, or external queries: out_of_scope.
6 Keep the student's exact words: typos, SMS, Kinyarwanda, as typed.


---

## Step 3: The dataset

**YOUR DOMAIN.** At least 30 messages per label. Sources, in order of value: your chatbot's UNMATCHED log and Step 10 local phrasing from Portfolio Project 1; at least 20 messages from real people you ask (classmates, family, friends, for example on WhatsApp: *how would you ask a bus desk about luggage?*); then your own. Keep every message exactly as it was typed.

In [4]:
# STEP 3. YOUR DOMAIN: the dataset. Each row is (message, label, source).
# source is one of: "rules", "unmatched", "local", "people", "written"
DATA = [
    # --- SIGN_IN (30 messages) ---
    ("sign me in for my exam", "sign_in", "rules"),
    ("i am here for the test", "sign_in", "people"),
    ("nshaka kwiyandikisha", "sign_in", "local"),
    ("signing in now", "sign_in", "written"),
    ("mark my attendance pls", "sign_in", "unmatched"),
    ("i have arrived for the exam", "sign_in", "written"),
    ("check me in", "sign_in", "people"),
    ("ndi hano", "sign_in", "local"),
    ("register my presence", "sign_in", "written"),
    ("can i sign in", "sign_in", "rules"),
    ("exam sign in please", "sign_in", "written"),
    ("i just entered the hall", "sign_in", "people"),
    ("injira mu bizamini", "sign_in", "local"),
    ("sgin in", "sign_in", "written"),
    ("attendance sign in", "sign_in", "unmatched"),
    ("ready for exam check in", "sign_in", "written"),
    ("i am present", "sign_in", "people"),
    ("vumbi sign me in", "sign_in", "written"),
    ("start my exam attendance", "sign_in", "written"),
    ("kwiyandikisha ku kizamini", "sign_in", "local"),
    ("logging my arrival", "sign_in", "written"),
    ("tap to sign in", "sign_in", "unmatched"),
    ("sign me up for test", "sign_in", "written"),
    ("i am at the venue to sign in", "sign_in", "people"),
    ("kwakira", "sign_in", "local"),
    ("pls sign me in", "sign_in", "written"),
    ("exam entry sign in", "sign_in", "written"),
    ("enregistrement", "sign_in", "unmatched"),
    ("i want to sign in now", "sign_in", "written"),
    ("confirm my presence", "sign_in", "written"),

    # --- SIGN_OUT (30 messages) ---
    ("sign me out", "sign_out", "rules"),
    ("i am done with my exam", "sign_out", "people"),
    ("ndasoje ikizamini", "sign_out", "local"),
    ("signing out", "sign_out", "written"),
    ("i am leaving the hall", "sign_out", "unmatched"),
    ("check me out", "sign_out", "written"),
    ("finished the test", "sign_out", "people"),
    ("taha", "sign_out", "local"),
    ("record my departure", "sign_out", "written"),
    ("can i sign out", "sign_out", "rules"),
    ("exam sign out please", "sign_out", "written"),
    ("i am exiting now", "sign_out", "people"),
    ("ndatashye", "sign_out", "local"),
    ("sgin out", "sign_out", "written"),
    ("attendance sign out", "sign_out", "unmatched"),
    ("done with exam check out", "sign_out", "written"),
    ("leaving the venue", "sign_out", "people"),
    ("sign me out pls", "sign_out", "written"),
    ("end my exam session", "sign_out", "written"),
    ("gusohoka mu kizamini", "sign_out", "local"),
    ("logging my exit", "sign_out", "written"),
    ("tap to sign out", "sign_out", "unmatched"),
    ("finish exam and sign out", "sign_out", "written"),
    ("i am leaving the room", "sign_out", "people"),
    ("ndashojije", "sign_out", "local"),
    ("pls sign me out", "sign_out", "written"),
    ("exam exit sign out", "sign_out", "written"),
    ("deconnexion", "sign_out", "unmatched"),
    ("i want to sign out now", "sign_out", "written"),
    ("confirm my exit", "sign_out", "written"),

    # --- CHECK_STATUS (30 messages) ---
    ("check my attendance status", "check_status", "rules"),
    ("did my sign in go through", "check_status", "people"),
    ("ese nanyuzeho?", "check_status", "local"),
    ("am i signed in", "check_status", "written"),
    ("show my status", "check_status", "unmatched"),
    ("check if i am recorded", "check_status", "written"),
    ("verify my attendance", "check_status", "people"),
    ("reba niba nanditswe", "check_status", "local"),
    ("is my sign in saved", "check_status", "written"),
    ("status check please", "check_status", "rules"),
    ("have i been marked present", "check_status", "written"),
    ("check my record", "check_status", "people"),
    ("amakuru yanjye", "check_status", "local"),
    ("chk status", "check_status", "written"),
    ("attendance log check", "check_status", "unmatched"),
    ("am i checked in properly", "check_status", "written"),
    ("verify my session", "check_status", "people"),
    ("show attendance record", "check_status", "written"),
    ("is my attendance active", "check_status", "written"),
    ("garagaza uko mbyaye", "check_status", "local"),
    ("confirm my tracking status", "check_status", "written"),
    ("did it record", "check_status", "unmatched"),
    ("check my profile status", "check_status", "written"),
    ("am i logged in", "check_status", "people"),
    ("reba status yanjye", "check_status", "local"),
    ("pls check status", "check_status", "written"),
    ("attendance verification", "check_status", "written"),
    ("suivi status", "check_status", "unmatched"),
    ("i want to see my status", "check_status", "written"),
    ("is my name on the list", "check_status", "written"),

    # --- VENUE_LOCATION (30 messages) ---
    ("where is my exam hall", "venue_location", "rules"),
    ("which room is the test in", "venue_location", "people"),
    ("icyumba cy'ikizamini kiri he?", "venue_location", "local"),
    ("exam venue location", "venue_location", "written"),
    ("where do i go for the exam", "venue_location", "unmatched"),
    ("find my exam room", "venue_location", "written"),
    ("what building is the test", "venue_location", "people"),
    ("aho ikizamini kibera", "venue_location", "local"),
    ("location of exam hall", "venue_location", "written"),
    ("where is the venue", "venue_location", "rules"),
    ("room number for exam", "venue_location", "written"),
    ("where should i sign in", "venue_location", "people"),
    ("icyumba nicaramo", "venue_location", "local"),
    ("vrnue location", "venue_location", "written"),
    ("exam hall map", "venue_location", "unmatched"),
    ("where is hall a", "venue_location", "written"),
    ("directions to exam room", "venue_location", "people"),
    ("show venue location", "venue_location", "written"),
    ("what is the room assignment", "venue_location", "written"),
    ("icyapa cy'aho dukorera", "venue_location", "local"),
    ("find exam building", "venue_location", "written"),
    ("where is my seat located", "venue_location", "unmatched"),
    ("locate exam venue", "venue_location", "written"),
    ("where is block b", "venue_location", "people"),
    ("hari he icyumba?", "venue_location", "local"),
    ("pls tell me the venue", "venue_location", "written"),
    ("exam hall guidance", "venue_location", "written"),
    ("localisation", "venue_location", "unmatched"),
    ("i am looking for my exam room", "venue_location", "written"),
    ("where do we write", "venue_location", "written"),

    # --- HELP (30 messages) ---
    ("i need help", "help", "rules"),
    ("how does this bot work", "help", "people"),
    ("fasha", "help", "local"),
    ("help me please", "help", "written"),
    ("what can you do", "help", "unmatched"),
    ("i am stuck", "help", "written"),
    ("guide me", "help", "people"),
    ("ndafite ikibazo", "help", "local"),
    ("support instructions", "help", "written"),
    ("how to use this system", "help", "rules"),
    ("system help menu", "help", "written"),
    ("give me assistance", "help", "people"),
    ("nyobora", "help", "local"),
    ("hlp", "help", "written"),
    ("bot commands", "help", "unmatched"),
    ("i need technical support", "help", "written"),
    ("how do i sign in here", "help", "people"),
    ("show help options", "help", "written"),
    ("what are the options", "help", "written"),
    ("mbwira uko bikora", "help", "local"),
    ("assist me", "help", "written"),
    ("how do i proceed", "help", "unmatched"),
    ("need guidance", "help", "written"),
    ("show instructions", "help", "people"),
    ("mfasha hano", "help", "local"),
    ("pls help", "help", "written"),
    ("help desk info", "help", "written"),
    ("aide", "help", "unmatched"),
    ("i don't understand", "help", "written"),
    ("how to track attendance", "help", "written"),

    # --- OUT_OF_SCOPE (30 messages) ---
    ("who won the football match", "out_of_scope", "rules"),
    ("what is the weather today", "out_of_scope", "people"),
    ("amakuru ya politiki ki?", "out_of_scope", "local"),
    ("order me a pizza", "out_of_scope", "written"),
    ("play music", "out_of_scope", "unmatched"),
    ("who is the president", "out_of_scope", "written"),
    ("what time is it", "out_of_scope", "people"),
    ("shaka akazi", "out_of_scope", "local"),
    ("tell me a joke", "out_of_scope", "written"),
    ("how do i cook rice", "out_of_scope", "rules"),
    ("flight tickets to nairobi", "out_of_scope", "written"),
    ("buy bitcoin", "out_of_scope", "people"),
    ("uririmba iki?", "out_of_scope", "local"),
    ("asdfgh", "out_of_scope", "written"),
    ("movie recommendations", "out_of_scope", "unmatched"),
    ("what is the capital of france", "out_of_scope", "written"),
    ("book a hotel room", "out_of_scope", "people"),
    ("rent a car", "out_of_scope", "written"),
    ("latest tech news", "out_of_scope", "written"),
    ("igiciro cy'inyama", "out_of_scope", "local"),
    ("stock market prices", "out_of_scope", "written"),
    ("can you write a poem", "out_of_scope", "unmatched"),
    ("gym schedule", "out_of_scope", "written"),
    ("recipe for cake", "out_of_scope", "people"),
    ("hari imvura hanze?", "out_of_scope", "local"),
    ("find me a job", "out_of_scope", "written"),
    ("crypto prices", "out_of_scope", "written"),
    ("lyrics of a song", "out_of_scope", "unmatched"),
    ("what is your name", "out_of_scope", "written"),
    ("hello there", "out_of_scope", "written"),
]
print(len(DATA), "rows")

180 rows


---

## Step 4: The quality report

**PROVIDED.** Read every line. Fix the data in Step 3 until every label is ok, there are no duplicates and no conflicts, and the balance is 1.5 or less. Then run Step 3 and Step 4 again.

In [5]:
# STEP 4. PROVIDED: the quality report. Read it, fix the data in Step 3, run again.
def clean(text):
    """Remove punctuation (keeping the colon of 21:00), lowercase, trim."""
    return re.sub(r"[^\w\s:]", "", text).lower().strip()


per_label = Counter()
per_source = Counter()
unknown = []
for text, label, source in DATA:
    per_label[label] += 1
    per_source[source] += 1
    if label not in INTENTS:
        unknown.append((text, label))

print("Per label:")
for label in INTENTS:
    n = per_label[label]
    note = "ok" if n >= TARGET else f"needs more, target {TARGET}"
    print(f"   {label:14}{n:4}   {note}")
print("Per source:", dict(per_source))
print("people:", per_source["people"], "of at least 20")

seen = {}
duplicates = []
conflicts = []
for text, label, source in DATA:
    key = clean(text)
    if key in seen:
        duplicates.append(text)
        if seen[key] != label:
            conflicts.append(text)
    seen[key] = label

print("Labels not in INTENTS:", unknown)
print("Duplicates:", duplicates)
print("Conflicting labels:", conflicts)
counts = [per_label[label] for label in INTENTS]
print("Balance, largest / smallest:", round(max(counts) / max(1, min(counts)), 2))

Per label:
   sign_in         30   ok
   sign_out        30   ok
   check_status    30   ok
   venue_location  30   ok
   help            30   ok
   out_of_scope    30   ok
Per source: {'rules': 12, 'people': 30, 'local': 30, 'written': 84, 'unmatched': 24}
people: 30 of at least 20
Labels not in INTENTS: []
Duplicates: []
Conflicting labels: []
Balance, largest / smallest: 1.0


---

## Step 5: The split, and the CSV file

**PROVIDED.** 20% of each label becomes the test set. The file `intents.csv` appears in the Files panel on the left: download it, and commit it to GitHub.

In [6]:
# STEP 5. PROVIDED: the 80/20 split, intent by intent, with a fixed seed; then the CSV.
rng = random.Random(8)

by_label = {}
for row in DATA:
    if row[1] not in by_label:
        by_label[row[1]] = []
    by_label[row[1]].append(row)

TRAIN = []
TEST = []
for label in sorted(by_label):
    rows = by_label[label][:]
    rng.shuffle(rows)
    cut = len(rows) // 5
    TEST.extend(rows[:cut])
    TRAIN.extend(rows[cut:])

with open("intents.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["text", "intent", "source", "split"])
    for text, label, source in TRAIN:
        writer.writerow([text, label, source, "train"])
    for text, label, source in TEST:
        writer.writerow([text, label, source, "test"])

df = pd.read_csv("intents.csv")
print(len(TRAIN), "training,", len(TEST), "test, saved to intents.csv")
print(pd.crosstab(df["intent"], df["split"]))

144 training, 36 test, saved to intents.csv
split           test  train
intent                     
check_status       6     24
help               6     24
out_of_scope       6     24
sign_in            6     24
sign_out           6     24
venue_location     6     24


---

## Step 6: The baseline: your rules on your test set

**YOUR DOMAIN.** Paste the rule table of your own chatbot. The baseline is the number your classifier must beat in Week 9. Every line it prints is a message the rules get wrong.

In [7]:
# STEP 6. YOUR DOMAIN: paste the RULES list for your Exam Attendance Tracker.
RULES = [
    ("sign_in", r"\bsign\s?in\b|\barriv\b|kwiyandikisha|ndi hano|present"),
    ("sign_out", r"\bsign\s?out\b|\bleav\b|\bexan?\b|ndasoje|taha"),
    ("check_status", r"\bstatus\b|\brecord\b|\bcheck\b|\bverify\b|nanyuzeho"),
    ("venue_location", r"\bwhere\b|\broom\b|\bhall\b|\blocation\b|icyumba"),
    ("help", r"\bhelp\b|\bassist\b|\bhow\b|fasha"),
]
print(len(RULES), "rules")

5 rules


In [8]:
# STEP 6, continued. PROVIDED: the baseline, your rules on your test set.
def rule_label(text):
    """The first rule that matches, or out_of_scope."""
    for label, pattern in RULES:
        if re.search(pattern, clean(text)):
            return label
    return "out_of_scope"


right = 0
for text, label, source in TEST:
    guess = rule_label(text)
    if guess == label:
        right += 1
    else:
        print(f"   {text!r}: labelled {label}, rules say {guess}")
print(f"Baseline: {right} of {len(TEST)} test messages right")

   'is my sign in saved': labelled check_status, rules say sign_in
   'garagaza uko mbyaye': labelled check_status, rules say out_of_scope
   'hlp': labelled help, rules say out_of_scope
   'what are the options': labelled help, rules say out_of_scope
   'i am stuck': labelled help, rules say out_of_scope
   'guide me': labelled help, rules say out_of_scope
   'signing in now': labelled sign_in, rules say out_of_scope
   'confirm my presence': labelled sign_in, rules say out_of_scope
   'logging my arrival': labelled sign_in, rules say out_of_scope
   'i have arrived for the exam': labelled sign_in, rules say out_of_scope
   'confirm my exit': labelled sign_out, rules say out_of_scope
   'logging my exit': labelled sign_out, rules say out_of_scope
   'done with exam check out': labelled sign_out, rules say check_status
   'i am leaving the room': labelled sign_out, rules say venue_location
   'i am exiting now': labelled sign_out, rules say out_of_scope
   'pls tell me the venue': labe

---

## Step 7: The entities of your domain

**YOUR DOMAIN.** All entity labels of your own, each with the closed list of names your users mention: places, products, services, days. The provided part counts how often they appear in your dataset.

In [9]:
# STEP 7. YOUR DOMAIN: the entities of your domain.
ENTITY_LISTS = {
    "BUILDING": ["block a", "block b", "hall a", "hall b"],
    "DAY": ["today", "tomorrow", "monday", "tuesday", "wednesday", "thursday", "friday"],
}
patterns = []
for label in ENTITY_LISTS:
    for name in ENTITY_LISTS[label]:
        patterns.append({"label": label, "pattern": [{"LOWER": name}]})

if "entity_ruler" in nlp.pipe_names:
    nlp.remove_pipe("entity_ruler")
ruler = nlp.add_pipe("entity_ruler", before="ner")
ruler.add_patterns(patterns)

found = Counter()
examples = []
for text, label, source in DATA:
    ents = [(e.text, e.label_) for e in nlp(text.lower()).ents]
    for e in ents:
        found[e[1]] += 1
    if ents and len(examples) < 5:
        examples.append((text, ents))

print("Entities found in the dataset:", dict(found))
for text, ents in examples:
    print("  ", text, "->", ents)

Entities found in the dataset: {'ORG': 8, 'PERSON': 16, 'DAY': 1, 'GPE': 1}
   nshaka kwiyandikisha -> [('kwiyandikisha', 'ORG')]
   mark my attendance pls -> [('mark my attendance pls', 'PERSON')]
   ndi hano -> [('ndi hano', 'PERSON')]
   injira mu bizamini -> [('injira mu bizamini', 'PERSON')]
   kwiyandikisha ku kizamini -> [('kwiyandikisha ku kizamini', 'PERSON')]


---

## Step 8: Publish and submit

1. In the repository of your Portfolio Project 1, create a folder named `dataset`.
2. Upload this notebook (File, then Download, then .ipynb) and `intents.csv` into it.
3. Add `dataset/README.md` from `README_template_dataset.md`, every section filled in.
4. Open the folder in a private browsing window. Paste its URL in Google Classroom and click
   **Turn in**, by **Monday 02 November 2026, 23:59**.

Never put real names, phone numbers or ID numbers in the dataset: it is public.